In [1]:

!pip install -q transformers torch pandas

import os
import torch
import json
import pandas as pd
from typing import List, Dict, Union, Any
from transformers import CLIPTokenizer, CLIPTextModel

print("Software environments and libraries successfully provisioned.")

Software environments and libraries successfully provisioned.


In [8]:
import os
import torch
import logging
from typing import List, Dict, Union, Any
from transformers import CLIPTokenizer, CLIPTextModel, logging as tf_logging

# Force Hugging Face to only log actual errors, silencing the weight mismatches
tf_logging.set_verbosity_error()

class TextPreprocessingEngine:
    def __init__(self, model_checkpoint: str = "openai/clip-vit-base-patch32", max_sequence_length: int = 77):
        self.model_checkpoint = model_checkpoint
        self.max_length = max_sequence_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"[BOOT] Initializing engine on device: {self.device.type.upper()}")
        try:
            self.tokenizer = CLIPTokenizer.from_pretrained(model_checkpoint)
            self.text_encoder = CLIPTextModel.from_pretrained(model_checkpoint).to(self.device)
            self.text_encoder.eval()
            print(f"[BOOT] Successfully loaded weights for {model_checkpoint}")
        except Exception as e:
            print(f"[CRITICAL ABORT] Dependency load failure: {str(e)}")
            raise

    def process_input(self, raw_prompts: Union[str, List[str]]) -> Dict[str, Any]:
        if isinstance(raw_prompts, str):
            raw_prompts = [raw_prompts]

        cleaned_prompts = [str(p).strip() for p in raw_prompts if len(str(p).strip()) > 0]
        if not cleaned_prompts:
            raise ValueError("[VALIDATION ERROR] Input batch contains no valid text.")

        print(f"[EXECUTION] Tokenizing and encoding batch of {len(cleaned_prompts)} items...")

        tokenized_inputs = self.tokenizer(
            cleaned_prompts,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = tokenized_inputs["input_ids"].to(self.device)
        attention_mask = tokenized_inputs["attention_mask"].to(self.device)

        with torch.no_grad():
            encoder_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            dense_embeddings = encoder_outputs.last_hidden_state

        payload_package = {
            "metadata": {
                "engine_model": self.model_checkpoint,
                "batch_size": len(cleaned_prompts),
                "tensor_max_sequence_length": self.max_length,
                "embedding_feature_depth": dense_embeddings.shape[2]
            },
            "raw_prompts": cleaned_prompts,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "embeddings": dense_embeddings
        }
        return payload_package

    def export_diagnostic_log(self, package: Dict[str, Any], file_name: str = "pipeline_config.json"):
        log_path = os.path.join(os.getcwd(), file_name)
        with open(log_path, 'w') as json_file:
            json.dump(package["metadata"], json_file, indent=4)
        print(f"[DISK EXPORT] Metadata profile written to: {log_path}")

print("[STATUS] Cell 2 Complete: Engine compiled cleanly.")

[STATUS] Cell 2 Complete: Engine compiled cleanly.


In [11]:
print("STARTING SOFTWARE DEPLOYMENT VALIDATION RUN\n")

app_engine = TextPreprocessingEngine()

production_prompts = [
    "Vibrant watercolor painting of an ancient stone castle sitting on top of a misty green hill",
    "Hyper-realistic macro photography shot of a pristine water lily petal with glowing dew drops at dawn",
    "Minimalist retro synthwave poster art featuring a pink neon sun rising behind dark grid lines"
]

try:
    runtime_payload = app_engine.process_input(production_prompts)
    meta_stats = runtime_payload["metadata"]

    print("\n" + "="*50)
    print("SOFTWARE PERFORMANCE & GRAPHICS MONITOR")
    print("="*50)
    print(f" 🔹 Core Transformer Framework  : {meta_stats['engine_model']}")
    print(f" 🔹 Synchronized Batch Load     : {meta_stats['batch_size']} items processed in parallel")
    print(f" 🔹 Generated Sequence Matrix   : {list(runtime_payload['embeddings'].shape)}")
    print(f"    ↳ [Batch Rows, Standardized Max Token Sequence, Vector Dimension Channels]")
    print(f" 🔹 Target Token Matrix Layout  : {list(runtime_payload['input_ids'].shape)}")
    print(f" 🔹 System Memory Allocation    : Storage mapped safely onto {runtime_payload['embeddings'].device.type.upper()}")
    print("="*50 + "\n")

    app_engine.export_diagnostic_log(runtime_payload)
    print("\n[STATUS] Cell 3 Complete: Integration testing validation successful!")

except Exception as pipeline_exception:
    print(f"[EMERGENCY TERMINATION] Software failed to process loop structures: {pipeline_exception}")

STARTING SOFTWARE DEPLOYMENT VALIDATION RUN

[BOOT] Initializing engine on device: CUDA


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[BOOT] Successfully loaded weights for openai/clip-vit-base-patch32
[EXECUTION] Tokenizing and encoding batch of 3 items...

SOFTWARE PERFORMANCE & GRAPHICS MONITOR
 🔹 Core Transformer Framework  : openai/clip-vit-base-patch32
 🔹 Synchronized Batch Load     : 3 items processed in parallel
 🔹 Generated Sequence Matrix   : [3, 77, 512]
    ↳ [Batch Rows, Standardized Max Token Sequence, Vector Dimension Channels]
 🔹 Target Token Matrix Layout  : [3, 77]
 🔹 System Memory Allocation    : Storage mapped safely onto CUDA

[DISK EXPORT] Metadata profile written to: /content/pipeline_config.json

[STATUS] Cell 3 Complete: Integration testing validation successful!


In [10]:
import pandas as pd

print("SUB-WORD TOKENIZER CONSOLE ANALYSIS")

sample_prompt_string = production_prompts[1]
encoded_ids_array = runtime_payload["input_ids"][1].cpu().numpy()

vocabulary_tokens = app_engine.tokenizer.convert_ids_to_tokens(encoded_ids_array)

tensor_inspection_df = pd.DataFrame({
    "Matrix Token Index Position": range(len(encoded_ids_array)),
    "Vocabulary String Token Tokenized": vocabulary_tokens,
    "Numerical Vocabulary ID Encoded": encoded_ids_array
})

print(f"\nTarget Input Sentence: \"{sample_prompt_string}\"\n")
print(tensor_inspection_df.head(15).to_string(index=False))
print("\n" + "."*60)
print(f"And {len(encoded_ids_array) - 15} more index dimensions padded cleanly with [<endoftext>] markers to lock sequence size constraint at 77...")
print("."*60)
print("\n Engineering telemetry verification complete.")

SUB-WORD TOKENIZER CONSOLE ANALYSIS

Target Input Sentence: "Hyper-realistic macro photography shot of a pristine water lily petal with glowing dew drops at dawn"

 Matrix Token Index Position Vocabulary String Token Tokenized  Numerical Vocabulary ID Encoded
                           0                   <|startoftext|>                            49406
                           1                         hyper</w>                            22146
                           2                             -</w>                              268
                           3                     realistic</w>                            16157
                           4                         macro</w>                            16626
                           5                   photography</w>                             2108
                           6                          shot</w>                             2000
                           7                            of</w>      